# 05 · Resultados Finales y Métricas del Modelo
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Isaac Oviedo  
> **Objetivo:** Consolidar todas las métricas, visualizaciones finales y reporte para el informe técnico.

---

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize

from src.preprocessing import preprocess
from src.models import get_cnn_representation, evaluate_model
from src.config import CLASS_LABELS

UR_RED = "#DA0921"; UR_NAVY = "#242839"; UR_TECH = "#0E6A8C"; UR_GREEN = "#1A6E3A"

data = preprocess("../data/evaluaciones_docentes.csv")
cnn_model    = tf.keras.models.load_model("../models/cnn_model.keras")
fusion_model = tf.keras.models.load_model("../models/fusion_model.keras")
with open("../models/rf_model.pkl", "rb") as f:
    rf_model = pickle.load(f)
from src.config import NUMERIC_FEATURES
print("✅ Modelos cargados")

In [ ]:
# Generar predicciones finales
repr_test       = get_cnn_representation(cnn_model, data.X_text_test)
rf_proba_test   = rf_model.predict_proba(data.X_num_test)
X_fusion_test   = np.concatenate([repr_test, rf_proba_test], axis=1)
fusion_proba    = fusion_model.predict(X_fusion_test, verbose=0)
fusion_pred     = np.argmax(fusion_proba, axis=1)

print(f"Evaluaciones en test: {len(data.y_test)}")

## 1 · Matriz de confusión - Modelo de Fusión

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(data.y_test, fusion_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_LABELS)
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Matriz de confusión - Fusión CNN + RF",
             fontsize=13, fontweight="bold", color=UR_NAVY)
plt.tight_layout()
plt.savefig("../outputs/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("Interpretación:")
for i, cls in enumerate(CLASS_LABELS):
    tp = cm[i, i]; total = cm[i].sum()
    print(f"  {cls:<12}: {tp}/{total} correctos ({tp/total*100:.1f}%)")

## 2 · Curvas ROC por clase

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
y_bin = label_binarize(data.y_test, classes=[0, 1, 2])
colors_roc = [UR_RED, UR_TECH, UR_GREEN]

for i, (cls, color) in enumerate(zip(CLASS_LABELS, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], fusion_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{cls} (AUC = {roc_auc_val:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
ax.set_xlabel("Tasa de Falsos Positivos"); ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("Curvas ROC por clase - Fusión CNN + RF",
             fontsize=13, fontweight="bold", color=UR_NAVY)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig("../outputs/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Reporte de clasificación completo

In [ ]:
from sklearn.metrics import classification_report
report = classification_report(
    data.y_test, fusion_pred,
    target_names=CLASS_LABELS, digits=4
)
print(report)

## 4 · Resumen ejecutivo para el informe técnico

In [ ]:
metrics = evaluate_model(data.y_test, fusion_pred, fusion_proba, "Fusión CNN + RF")

print("=" * 60)
print("  RESUMEN EJECUTIVO - EduPredict")
print("  Equipo Sin Convergencia · SIC 2025")
print("=" * 60)
acc_v  = metrics["accuracy"]
f1_v   = metrics["f1_macro"]
auc_m  = metrics["auc_roc_macro"]
f1_r   = metrics["f1_per_class"]["En riesgo"]
f1_est = metrics["f1_per_class"]["Estable"]
f1_mej = metrics["f1_per_class"]["Mejora"]
print("  DATASET: 3.000 registros | 50 docentes | 8 semestres | 7 asignaturas")
print("  ARQUITECTURA: CNN-1D + Random Forest + Fusion Dense(32)+Dropout(0.3)")
print(f"  Accuracy : {acc_v:.4f}  |  F1-macro: {f1_v:.4f}  |  AUC-ROC: {auc_m:.4f}")
print(f"  F1 clase -> En riesgo: {f1_r:.4f} | Estable: {f1_est:.4f} | Mejora: {f1_mej:.4f}")
print("  Anti-overfitting: Dropout + EarlyStopping + L2 + CV5fold + class_weight balanced")

# Guardar resultados
results = {"metrics_fusion": metrics}
with open("../outputs/training_results.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("  ✅ Resultados guardados en outputs/training_results.json")

## 5 · Archivos generados

| Archivo | Descripción |
|---|---|
| `outputs/eda_01_target.png` | Distribución de clases y evolución temporal |
| `outputs/eda_02_scores.png` | Boxplots de puntajes por clase |
| `outputs/eda_03_corr.png` | Matriz de correlación |
| `outputs/eda_04_text.png` | Señales léxicas por clase |
| `outputs/rf_cv_results.png` | Resultados de validación cruzada RF |
| `outputs/rf_feature_importance.png` | Importancia de features RF |
| `outputs/cnn_learning_curves.png` | Curvas de aprendizaje CNN |
| `outputs/fusion_learning_curves.png` | Curvas de aprendizaje Fusión |
| `outputs/ablation_study.png` | Comparación de los 3 modelos |
| `outputs/confusion_matrix.png` | Matriz de confusión final |
| `outputs/roc_curves.png` | Curvas ROC por clase |
| `outputs/training_results.json` | Métricas en JSON |

> ✅ **Pipeline completo terminado** - lanzar `streamlit run app/app.py` para el dashboard
